# 전체 판례 대상 테스트

In [1]:
import os
import glob
import json
import pandas as pd
import numpy as np
import time
from tqdm import tqdm
from konlpy.tag import Mecab
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score

# =========================================================
# [설정] 실험 조건 설정
# =========================================================
# [수정됨] 학습 데이터 폴더를 리스트로 설정 (대법원 + 하위법원)
DATA_DIRS = ["data/대법원_판례", "data/하위법원_판례"] 

# ★ 분석 대상 폴더 목록 (불용어 리스트가 있는 곳)
TARGET_STOPWORD_DIRS = ["stopword_공통", "stopword_판례", "stopword_법령"]

# ★ 중요: 몇 개의 사건 종류로 분류할지 설정 (10 or 20 추천)
TOP_N_CLASSES = 10

# ★ 시간 측정 신뢰도를 위한 반복 횟수 설정
NUM_ITERATIONS = 5 

print(f">>> 1. 데이터셋 로딩 및 복구 시작...")
print(f"    - 데이터 소스: {DATA_DIRS}")
print(f"    - 타겟 사건 수: Top {TOP_N_CLASSES if TOP_N_CLASSES else 'All'}")

def load_and_split_data(data_dirs_list, top_n=None):
    all_files = []
    
    # [수정됨] 여러 폴더에서 파일 리스트 수집
    for d in data_dirs_list:
        files = glob.glob(os.path.join(d, "**", "*.json"), recursive=True)
        all_files.extend(files)
        print(f"   - '{d}' 폴더 스캔: {len(files)}개 파일 발견")

    texts = []
    labels = []
    
    # 데이터 로딩
    for file_path in tqdm(all_files, desc="Total JSON Loading"):
        try:
            with open(file_path, 'r', encoding='utf-8') as f:
                data = json.load(f)
                content = data.get('판례내용', "")
                label = data.get('사건명', "")
                
                if content and label:
                    # 전처리: 줄바꿈 제거 등
                    content = content.replace("<br/>", " ").replace("\n", " ")
                    texts.append(content)
                    labels.append(label)
        except:
            continue
    
    df = pd.DataFrame({'text': texts, 'label': labels})
    
    # 설정된 Top N 만큼 필터링
    if top_n:
        target_classes = df['label'].value_counts().head(top_n).index.tolist()
        df_filtered = df[df['label'].isin(target_classes)].copy()
        print(f"   - 대상 사건명(Top {top_n}): {target_classes}")
    else:
        df_filtered = df.copy()
        print(f"   - 대상 사건명: 전체 ({df['label'].nunique()}개)")
    
    print(f"   - 전체 데이터(통합): {len(df)}건")
    print(f"   - 필터링 후 사용 데이터: {len(df_filtered)}건")

    return train_test_split(
        df_filtered['text'], df_filtered['label'], 
        test_size=0.2, random_state=42, stratify=df_filtered['label']
    )

try:
    # 함수 호출 시 리스트(DATA_DIRS) 전달
    X_train, X_test, y_train, y_test = load_and_split_data(DATA_DIRS, TOP_N_CLASSES)
except ValueError:
    print("[Error] 데이터를 찾을 수 없습니다. 경로를 확인해주세요.")
    # 테스트용 더미 데이터
    X_train, y_train = ["테스트"] * 10, ["사기"] * 10
    X_test, y_test = ["테스트"] * 2, ["사기"] * 2

# =========================================================
# [Part 2] 불용어 리스트 복구 (다중 폴더 지원 + 파일명 호환성)
# =========================================================
print("\n>>> 2. 불용어 리스트 파일 로딩 (다중 소스)...")

def load_stopword_flexible(folder, suffix):
    """
    폴더 내에서 파일명을 유연하게 찾아서 로드함
    (예: 'stopwords_exp_A_zipf.csv' 또는 'candidates_A_zipf.csv')
    """
    prefixes = ["stopwords_exp_", "candidates_"] # 호환될 접두사 목록
    
    for pre in prefixes:
        filename = f"{pre}{suffix}"
        path = os.path.join(folder, filename)
        if os.path.exists(path):
            try:
                return set(pd.read_csv(path)['word'].astype(str).tolist())
            except:
                continue
    return None

# 일반 불용어 로드
GENERAL_STOPWORD_FILE = "stopwords.txt"
if not os.path.exists(GENERAL_STOPWORD_FILE):
    with open(GENERAL_STOPWORD_FILE, "w", encoding="utf-8") as f:
        f.write("이\n그\n저\n것\n수\n등\n들\n및\n에\n의\n를\n가\n은\n는")

def load_txt_stopwords(filepath):
    with open(filepath, 'r', encoding='utf-8') as f:
        return set([line.strip() for line in f if line.strip()])

set_general = load_txt_stopwords(GENERAL_STOPWORD_FILE)
print(f"   - 일반 불용어(General) 로드 완료: {len(set_general)}개")

# =========================================================
# [Part 3] 실험 시나리오 자동 생성
# =========================================================
stopword_sets = {
    "1. Baseline (None)": set(),
    "2. General Only": set_general
}

# 폴더별 순회
for folder in TARGET_STOPWORD_DIRS:
    if not os.path.exists(folder):
        print(f"   [Skip] '{folder}' 폴더가 없습니다.")
        continue
    
    print(f"   - '{folder}' 폴더 스캔 중...")
    
    # 각 기법별 파일 로드 (파일명 접미사만 지정)
    s_a = load_stopword_flexible(folder, "A_zipf.csv")
    s_b = load_stopword_flexible(folder, "B_entropy.csv")
    s_c = load_stopword_flexible(folder, "C_mpvp.csv")
    s_e = load_stopword_flexible(folder, "E_intersection.csv")
    
    # 폴더명을 태그로 붙여서 실험군 추가
    tag = f"[{folder.replace('stopword_', '')}]"
    
    if s_a: stopword_sets[f"{tag} Exp A (Zipf)"] = s_a
    if s_b: stopword_sets[f"{tag} Exp B (Ent)"] = s_b
    if s_c: stopword_sets[f"{tag} Exp C (MPVP)"] = s_c
    if s_e: 
        stopword_sets[f"{tag} Exp E (Intersect)"] = s_e
        # Hybrid (General + E) 추가 -> 성능 챔피언 후보
        stopword_sets[f"{tag} Hybrid (Gen+E)"] = set_general.union(s_e)

print(f"   >>> 총 {len(stopword_sets)}개의 실험 시나리오가 준비되었습니다.")

# =========================================================
# [Part 4] 대규모 벤치마크 실행 (Robust Timing)
# =========================================================
print(f"\n>>> 3. [확장 실험] 판례 분류 성능 비교 (각 {NUM_ITERATIONS}회 반복)")
mecab = Mecab()

def tokenizer(text):
    # 명사만 추출
    return [word for word, pos in mecab.pos(text) if pos.startswith('N')]

results = []
print("-" * 100)
print(f"{'Experiment':<35} | {'Acc':<8} | {'F1':<8} | {'Avg Time':<10} | {'Count'}")
print("-" * 100)

for name, stop_set in stopword_sets.items():
    stop_list = list(stop_set) if stop_set else None
    
    # 시간 측정용 리스트
    times = []
    accuracies = []
    f1s = []
    
    # 반복 실행 Loop
    for i in range(NUM_ITERATIONS):
        pipeline = Pipeline([
            ('vect', TfidfVectorizer(tokenizer=tokenizer, stop_words=stop_list, max_features=5000)),
            ('clf', LogisticRegression(C=5.0, max_iter=1000, n_jobs=-1)) 
        ])
        
        start_time = time.time()
        try:
            pipeline.fit(X_train, y_train)
            pred = pipeline.predict(X_test)
            
            elapsed = time.time() - start_time
            acc = accuracy_score(y_test, pred)
            f1 = f1_score(y_test, pred, average='weighted')
            
            times.append(elapsed)
            accuracies.append(acc)
            f1s.append(f1)
            
        except Exception as e:
            print(f"{name:<35} | ERROR: {e}")
            break
            
    if times:
        avg_time = np.mean(times)
        std_time = np.std(times)
        avg_acc = np.mean(accuracies)
        avg_f1 = np.mean(f1s)
        
        print(f"{name:<35} | {avg_acc:.4f}   | {avg_f1:.4f}   | {avg_time:.3f}s (±{std_time:.2f}) | {len(stop_set)}")
        
        results.append({
            'Experiment': name,
            'Count': len(stop_set),
            'Accuracy': avg_acc,
            'F1_Score': avg_f1,
            'Avg_Time': avg_time,
            'Time_Std_Dev': std_time
        })

# =========================================================
# [Part 5] 결과 저장 (소수점 4자리 반올림)
# =========================================================
if results:
    # 1. 정확도 순으로 정렬
    final_df = pd.DataFrame(results).sort_values(by='Accuracy', ascending=False)
    
    # 2. 소수점 4째 자리 반올림
    final_df = final_df.round(4)
    
    print("\n[최종 종합 순위 (5회 평균)]")
    print(final_df)

    filename = f"benchmark_results_unified_top{TOP_N_CLASSES}.csv"
    final_df.to_csv(filename, index=False, encoding='utf-8-sig')
    print(f"\n결과 저장 완료: {filename}")

>>> 1. 데이터셋 로딩 및 복구 시작...
    - 데이터 소스: ['data/대법원_판례', 'data/하위법원_판례']
    - 타겟 사건 수: Top 10
   - 'data/대법원_판례' 폴더 스캔: 5501개 파일 발견
   - 'data/하위법원_판례' 폴더 스캔: 27729개 파일 발견


Total JSON Loading: 100%|██████████| 33230/33230 [00:19<00:00, 1734.43it/s]
/Users/nojaegyeong/anaconda3/envs/nlp_learning/lib/python3.11/site-packages/sklearn/feature_extraction/text.py:521: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


   - 대상 사건명(Top 10): ['손해배상(기)', '손해배상청구사건', '부당이득금', '구상금', '소유권이전등기', '배당이의', '임금', '법인세부과처분취소', '보험금', '소유권이전등기말소청구사건']
   - 전체 데이터(통합): 33230건
   - 필터링 후 사용 데이터: 4763건

>>> 2. 불용어 리스트 파일 로딩 (다중 소스)...
   - 일반 불용어(General) 로드 완료: 600개
   - 'stopword_공통' 폴더 스캔 중...
   - 'stopword_판례' 폴더 스캔 중...
   - 'stopword_법령' 폴더 스캔 중...
   >>> 총 17개의 실험 시나리오가 준비되었습니다.

>>> 3. [확장 실험] 판례 분류 성능 비교 (각 5회 반복)
----------------------------------------------------------------------------------------------------
Experiment                          | Acc      | F1       | Avg Time   | Count
----------------------------------------------------------------------------------------------------


/Users/nojaegyeong/anaconda3/envs/nlp_learning/lib/python3.11/site-packages/sklearn/feature_extraction/text.py:521: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(
/Users/nojaegyeong/anaconda3/envs/nlp_learning/lib/python3.11/site-packages/sklearn/feature_extraction/text.py:521: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(
/Users/nojaegyeong/anaconda3/envs/nlp_learning/lib/python3.11/site-packages/sklearn/feature_extraction/text.py:521: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(
/Users/nojaegyeong/anaconda3/envs/nlp_learning/lib/python3.11/site-packages/sklearn/feature_extraction/text.py:521: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


1. Baseline (None)                  | 0.8772   | 0.8746   | 26.104s (±1.71) | 0


/Users/nojaegyeong/anaconda3/envs/nlp_learning/lib/python3.11/site-packages/sklearn/feature_extraction/text.py:521: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(
/Users/nojaegyeong/anaconda3/envs/nlp_learning/lib/python3.11/site-packages/sklearn/feature_extraction/text.py:406: UserWarning: Your stop_words may be inconsistent with your preprocessing. Tokenizing the stop words generated tokens ['간', '개의', '거니', '걸', '겁', '견지', '결과', '결론', '고려', '곳', '공동', '관계', '관련', '구체', '구토', '근거', '기점', '기준', '김', '까닭', '나다', '나머지', '남', '누가', '대', '대로', '데', '도달', '도착', '동시', '뒤', '듯', '따름', '때문', '라', '마음', '만', '말', '메', '바', '밖', '반', '방면', '번', '부류', '불문', '비교', '뿐', '사람', '상대', '생각', '서술', '수', '습', '시작', '시초', '쓰', '아래', '아무', '앞', '연관', '옆', '외', '위', '윗', '의거', '의지', '이외', '이용', '이유', '일반', '일지', '임', '입각', '입장', '점', '정도', '제외', '종합', '주저', '줄', '중', '즈음', '지경', '진짜', '집', '쪽', '천', '총', '켠', '편', '한적', '할만', '항목', '해', '형식', '후'

... (27 outputs left out)

/Users/nojaegyeong/anaconda3/envs/nlp_learning/lib/python3.11/site-packages/sklearn/feature_extraction/text.py:521: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(
/Users/nojaegyeong/anaconda3/envs/nlp_learning/lib/python3.11/site-packages/sklearn/feature_extraction/text.py:521: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(
/Users/nojaegyeong/anaconda3/envs/nlp_learning/lib/python3.11/site-packages/sklearn/feature_extraction/text.py:521: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(
/Users/nojaegyeong/anaconda3/envs/nlp_learning/lib/python3.11/site-packages/sklearn/feature_extraction/text.py:521: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(
/Users/nojaegyeong/anaconda3/envs/nlp_learning/lib/python3.11/site-packages/sklearn/feature_extraction/t

[법령] Exp E (Intersect)              | 0.8741   | 0.8714   | 31.139s (±12.62) | 48


/Users/nojaegyeong/anaconda3/envs/nlp_learning/lib/python3.11/site-packages/sklearn/feature_extraction/text.py:521: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(
/Users/nojaegyeong/anaconda3/envs/nlp_learning/lib/python3.11/site-packages/sklearn/feature_extraction/text.py:406: UserWarning: Your stop_words may be inconsistent with your preprocessing. Tokenizing the stop words generated tokens ['간', '개의', '거니', '걸', '겁', '견지', '결과', '결론', '고려', '곳', '공동', '구체', '구토', '근거', '기점', '김', '까닭', '나다', '나머지', '남', '누가', '대', '대로', '데', '도달', '도착', '동시', '뒤', '듯', '따름', '때문', '라', '마음', '만', '말', '메', '바', '밖', '반', '방면', '번', '부류', '불문', '비교', '뿐', '사람', '상대', '생각', '서술', '수', '습', '시작', '시초', '쓰', '아래', '아무', '앞', '연관', '옆', '외', '위', '윗', '의거', '의지', '이외', '이용', '이유', '일반', '일지', '임', '입각', '입장', '점', '정도', '제외', '종합', '주저', '줄', '중', '즈음', '지경', '진짜', '집', '쪽', '천', '총', '켠', '편', '한적', '할만', '항목', '해', '형식', '후', '힘'] not in stop

[법령] Hybrid (Gen+E)                 | 0.8741   | 0.8715   | 25.363s (±0.34) | 643

[최종 종합 순위 (5회 평균)]
                Experiment  Count  Accuracy  F1_Score  Avg_Time  Time_Std_Dev
0       1. Baseline (None)      0    0.8772    0.8746   26.1044        1.7101
4        [공통] Exp C (MPVP)     54    0.8762    0.8736   26.7371        0.1828
1          2. General Only    600    0.8751    0.8727   26.4127        0.4838
13        [법령] Exp B (Ent)     77    0.8751    0.8725   19.9662        0.3408
5   [공통] Exp E (Intersect)     44    0.8751    0.8726   26.7583        0.2513
15  [법령] Exp E (Intersect)     48    0.8741    0.8714   31.1385       12.6203
16     [법령] Hybrid (Gen+E)    643    0.8741    0.8715   25.3625        0.3401
9        [판례] Exp C (MPVP)    143    0.8730    0.8711   19.4686        0.3718
12       [법령] Exp A (Zipf)     90    0.8730    0.8710   19.9929        0.7933
14       [법령] Exp C (MPVP)    129    0.8720    0.8692  104.0535      169.4234
6      [공통] Hybrid (Gen+E)    639    0.8

# 법령 대상 테스트

In [32]:
import os
import glob
import pandas as pd
import numpy as np
import time
import unicodedata
from tqdm import tqdm
from konlpy.tag import Mecab
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score

# =========================================================
# [설정] 데이터 경로 및 실험 옵션
# =========================================================
TEXT_DIR = "data/법률_정제_완료_txt화" 
META_FILE = "data/전체_법률_목록(2025-12-01).csv"

TARGET_STOPWORD_DIRS = ["stopword_공통", "stopword_법령", "stopword_판례"]
TOP_N_CLASSES = 10

# ★ 시간 측정 신뢰도를 위한 반복 횟수 설정
NUM_ITERATIONS = 5

# =========================================================
# [Part 1] 데이터셋 구축
# =========================================================
print(f">>> 1. 법령 데이터셋 구축 시작...")
print(f"    - 텍스트 폴더 경로: {os.path.abspath(TEXT_DIR)}")

def load_statute_data(text_dir, meta_file, top_n=10):
    try:
        try:
            meta_df = pd.read_csv(meta_file, encoding='cp949')
        except:
            meta_df = pd.read_csv(meta_file, encoding='utf-8')
    except Exception as e:
        print(f"[Error] 메타파일 로드 실패: {e}")
        return None, None, None, None

    label_map = dict(zip(meta_df['법령명'].str.strip(), meta_df['법령분야명'].str.strip()))
    
    texts = []
    labels = []
    txt_files = glob.glob(os.path.join(text_dir, "**", "*.txt"), recursive=True)
    
    for file_path in tqdm(txt_files, desc="Matching"):
        filename_raw = os.path.basename(file_path).replace(".txt", "").strip()
        # Mac OS 호환성을 위해 정규화
        filename = unicodedata.normalize('NFC', filename_raw)
        
        if filename in label_map:
            label = label_map[filename]
            if pd.isna(label) or not str(label).strip(): continue
            try:
                with open(file_path, 'r', encoding='utf-8') as f:
                    content = f.read()
                texts.append(content)
                labels.append(label)
            except: continue
    
    df = pd.DataFrame({'text': texts, 'label': labels})
    if len(df) == 0: return [], [], [], []

    if top_n:
        target_classes = df['label'].value_counts().head(top_n).index.tolist()
        df_filtered = df[df['label'].isin(target_classes)].copy()
    else:
        df_filtered = df.copy()
    
    print(f"   - 최종 학습 데이터: {len(df_filtered)}건")
    return train_test_split(df_filtered['text'], df_filtered['label'], test_size=0.2, random_state=42, stratify=df_filtered['label'])

try:
    X_train, X_test, y_train, y_test = load_statute_data(TEXT_DIR, META_FILE, TOP_N_CLASSES)
    if len(X_train) == 0: raise ValueError("Empty Data")
except Exception as e:
    print(f"[Warning] 데이터 로드 실패({e}). 테스트용 더미 데이터를 생성합니다.")
    X_train, y_train = ["제1조..."]*50, ["행정"]*50
    X_test, y_test = ["제1조..."]*10, ["행정"]*10

# =========================================================
# [Part 2] 불용어 리스트 로드 (파일명 유연성 추가)
# =========================================================
print("\n>>> 2. 실험용 불용어 리스트 준비...")

def load_stopword_flexible(folder, suffix):
    """
    폴더 내에서 파일명을 유연하게 찾아서 로드함
    (예: 'stopwords_exp_A_zipf.csv' 또는 'candidates_A_zipf.csv')
    """
    prefixes = ["stopwords_exp_", "candidates_"] # 호환될 접두사 목록
    
    for pre in prefixes:
        filename = f"{pre}{suffix}"
        path = os.path.join(folder, filename)
        if os.path.exists(path):
            try:
                return set(pd.read_csv(path)['word'].astype(str).tolist())
            except:
                continue
    return None

GENERAL_FILE = "stopwords.txt"
if not os.path.exists(GENERAL_FILE):
    with open(GENERAL_FILE, "w", encoding='utf-8') as f: f.write("이\n그\n저")
def load_txt(path):
    with open(path, 'r', encoding='utf-8') as f: return set([l.strip() for l in f if l.strip()])
set_general = load_txt(GENERAL_FILE)

stopword_sets = {
    "1. Baseline (None)": set(),
    "2. General Only": set_general
}

for folder in TARGET_STOPWORD_DIRS:
    if not os.path.exists(folder): continue
    tag = f"[{folder.replace('stopword_', '')}]"
    
    # 각 기법별 파일 로드 (유연한 로더 사용)
    s_a = load_stopword_flexible(folder, "A_zipf.csv")
    s_b = load_stopword_flexible(folder, "B_entropy.csv")
    s_c = load_stopword_flexible(folder, "C_mpvp.csv")
    s_e = load_stopword_flexible(folder, "E_intersection.csv")
    
    if s_a: stopword_sets[f"{tag} Exp A (Zipf)"] = s_a
    if s_b: stopword_sets[f"{tag} Exp B (Ent)"] = s_b
    if s_c: stopword_sets[f"{tag} Exp C (MPVP)"] = s_c
    if s_e:
        stopword_sets[f"{tag} Exp E (Intersect)"] = s_e
        stopword_sets[f"{tag} Hybrid (Gen+E)"] = set_general.union(s_e)

print(f"   >>> 총 {len(stopword_sets)}개의 실험 시나리오가 준비되었습니다.")

# =========================================================
# [Part 3] 반복 벤치마크 실행 (Robust Timing)
# =========================================================
print(f"\n>>> 3. [법령 분류] 성능 비교 (각 {NUM_ITERATIONS}회 반복)")
mecab = Mecab()
def tokenizer(text):
    return [word for word, pos in mecab.pos(text) if pos.startswith('N')]

results = []
print("-" * 100)
print(f"{'Experiment':<35} | {'Acc':<8} | {'F1':<8} | {'Avg Time':<10} | {'Count'}")
print("-" * 100)

for name, stop_set in stopword_sets.items():
    stop_list = list(stop_set) if stop_set else None
    
    times = []
    accuracies = []
    f1s = []
    
    for i in range(NUM_ITERATIONS):
        pipeline = Pipeline([
            ('vect', TfidfVectorizer(tokenizer=tokenizer, stop_words=stop_list, max_features=5000)),
            ('clf', LogisticRegression(C=5.0, max_iter=1000, n_jobs=-1)) 
        ])
        
        start_time = time.time()
        try:
            pipeline.fit(X_train, y_train)
            pred = pipeline.predict(X_test)
            
            elapsed = time.time() - start_time
            acc = accuracy_score(y_test, pred)
            f1 = f1_score(y_test, pred, average='weighted')
            
            times.append(elapsed)
            accuracies.append(acc)
            f1s.append(f1)
            
        except Exception as e:
            print(f"{name:<35} | ERROR: {e}")
            break
    
    if times:
        avg_time = np.mean(times)
        std_time = np.std(times)
        avg_acc = np.mean(accuracies)
        avg_f1 = np.mean(f1s)
        
        print(f"{name:<35} | {avg_acc:.4f}   | {avg_f1:.4f}   | {avg_time:.3f}s (±{std_time:.2f}) | {len(stop_set)}")
        
        results.append({
            'Experiment': name, 
            'Count': len(stop_set), 
            'Accuracy': avg_acc, 
            'F1_Score': avg_f1, 
            'Avg_Time': avg_time,
            'Time_Std_Dev': std_time
        })

# =========================================================
# [Part 4] 결과 저장
# =========================================================
if results:
    final_df = pd.DataFrame(results).sort_values(by='Accuracy', ascending=False).round(4)
    print("\n[최종 종합 순위 (5회 평균)]")
    print(final_df)
    
    filename = f"benchmark_results_statute_top{TOP_N_CLASSES}_robust.csv"
    final_df.to_csv(filename, index=False, encoding='utf-8-sig')
    print(f"\n결과 저장 완료: {filename}")

>>> 1. 법령 데이터셋 구축 시작...
    - 텍스트 폴더 경로: /Users/nojaegyeong/Documents/GitHub/Ko-legal-stopwords/data/법률_정제_완료_txt화


Matching: 100%|██████████| 1690/1690 [00:00<00:00, 1899.37it/s]
/Users/nojaegyeong/anaconda3/envs/nlp_learning/lib/python3.11/site-packages/sklearn/feature_extraction/text.py:521: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


   - 최종 학습 데이터: 486건

>>> 2. 실험용 불용어 리스트 준비...
   >>> 총 17개의 실험 시나리오가 준비되었습니다.

>>> 3. [법령 분류] 성능 비교 (각 5회 반복)
----------------------------------------------------------------------------------------------------
Experiment                          | Acc      | F1       | Avg Time   | Count
----------------------------------------------------------------------------------------------------


/Users/nojaegyeong/anaconda3/envs/nlp_learning/lib/python3.11/site-packages/sklearn/feature_extraction/text.py:521: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(
/Users/nojaegyeong/anaconda3/envs/nlp_learning/lib/python3.11/site-packages/sklearn/feature_extraction/text.py:521: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(
/Users/nojaegyeong/anaconda3/envs/nlp_learning/lib/python3.11/site-packages/sklearn/feature_extraction/text.py:521: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(
/Users/nojaegyeong/anaconda3/envs/nlp_learning/lib/python3.11/site-packages/sklearn/feature_extraction/text.py:521: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


1. Baseline (None)                  | 0.7857   | 0.7871   | 6.264s (±0.08) | 0


/Users/nojaegyeong/anaconda3/envs/nlp_learning/lib/python3.11/site-packages/sklearn/feature_extraction/text.py:521: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(
/Users/nojaegyeong/anaconda3/envs/nlp_learning/lib/python3.11/site-packages/sklearn/feature_extraction/text.py:406: UserWarning: Your stop_words may be inconsistent with your preprocessing. Tokenizing the stop words generated tokens ['간', '개의', '거니', '걸', '겁', '견지', '결과', '결론', '고려', '곳', '공동', '관계', '관련', '구체', '구토', '근거', '기점', '기준', '김', '까닭', '나다', '나머지', '남', '누가', '대', '대로', '데', '도달', '도착', '동시', '뒤', '듯', '따름', '때문', '라', '마음', '만', '말', '메', '바', '밖', '반', '방면', '번', '부류', '불문', '비교', '뿐', '사람', '상대', '생각', '서술', '수', '습', '시작', '시초', '쓰', '아래', '아무', '앞', '연관', '옆', '외', '위', '윗', '의거', '의지', '이외', '이용', '이유', '일반', '일지', '임', '입각', '입장', '점', '정도', '제외', '종합', '주저', '줄', '중', '즈음', '지경', '진짜', '집', '쪽', '천', '총', '켠', '편', '한적', '할만', '항목', '해', '형식', '후'

... (27 outputs left out)

/Users/nojaegyeong/anaconda3/envs/nlp_learning/lib/python3.11/site-packages/sklearn/feature_extraction/text.py:521: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(
/Users/nojaegyeong/anaconda3/envs/nlp_learning/lib/python3.11/site-packages/sklearn/feature_extraction/text.py:521: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(
/Users/nojaegyeong/anaconda3/envs/nlp_learning/lib/python3.11/site-packages/sklearn/feature_extraction/text.py:521: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(
/Users/nojaegyeong/anaconda3/envs/nlp_learning/lib/python3.11/site-packages/sklearn/feature_extraction/text.py:521: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(
/Users/nojaegyeong/anaconda3/envs/nlp_learning/lib/python3.11/site-packages/sklearn/feature_extraction/t

[판례] Exp E (Intersect)              | 0.7857   | 0.7871   | 6.302s (±0.05) | 62


/Users/nojaegyeong/anaconda3/envs/nlp_learning/lib/python3.11/site-packages/sklearn/feature_extraction/text.py:521: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(
/Users/nojaegyeong/anaconda3/envs/nlp_learning/lib/python3.11/site-packages/sklearn/feature_extraction/text.py:406: UserWarning: Your stop_words may be inconsistent with your preprocessing. Tokenizing the stop words generated tokens ['간', '개의', '거니', '걸', '겁', '견지', '결과', '결론', '고려', '곳', '공동', '구체', '구토', '근거', '기점', '기준', '김', '까닭', '나다', '나머지', '남', '누가', '대', '대로', '데', '도달', '도착', '동시', '뒤', '듯', '따름', '때문', '라', '마음', '만', '말', '메', '바', '밖', '반', '방면', '번', '부류', '불문', '비교', '뿐', '사람', '상대', '생각', '서술', '수', '습', '시작', '시초', '쓰', '아래', '아무', '앞', '연관', '옆', '외', '위', '윗', '의거', '의지', '이외', '이용', '일반', '일지', '임', '입각', '입장', '점', '정도', '제외', '종합', '주저', '줄', '중', '즈음', '지경', '진짜', '집', '쪽', '천', '총', '켠', '편', '한적', '할만', '항목', '해', '형식', '후', '힘'] not in stop

[판례] Hybrid (Gen+E)                 | 0.7857   | 0.7871   | 6.205s (±0.02) | 657

[최종 종합 순위 (5회 평균)]
                Experiment  Count  Accuracy  F1_Score  Avg_Time  Time_Std_Dev
9        [법령] Exp C (MPVP)    129    0.8061    0.8092    5.1946        0.0071
7        [법령] Exp A (Zipf)     90    0.8061    0.8073    6.2147        0.0154
0       1. Baseline (None)      0    0.7857    0.7871    6.2639        0.0820
15  [판례] Exp E (Intersect)     62    0.7857    0.7871    6.3016        0.0453
14       [판례] Exp C (MPVP)    143    0.7857    0.7882    6.2580        0.0160
13        [판례] Exp B (Ent)     83    0.7857    0.7871    6.2771        0.0107
12       [판례] Exp A (Zipf)    132    0.7857    0.7871    6.2711        0.0284
11     [법령] Hybrid (Gen+E)    643    0.7857    0.7875    6.2313        0.0156
10  [법령] Exp E (Intersect)     48    0.7857    0.7876    6.3028        0.0392
8         [법령] Exp B (Ent)     77    0.7857    0.7876    6.2693        0.0111
1          2. General Only    600    0.78